<div>
    <img src="https://storage.googleapis.com/kaggle-datasets-images/312305/633246/752964d08f6001573444649668b0b011/dataset-cover.jpg?t=2019-08-22-03-58-44" class="Header_CoverImg-sc-1431b7d ibFJYv">
</div>

In [ ]:
import keras
from keras import regularizers, optimizers
from keras import losses
from keras.models import Sequential, Model, load_model
from keras.layers import Dense, Input, Dropout, Embedding, LSTM
from keras.optimizers import RMSprop, Adam, Nadam
from keras.preprocessing import sequence

import torch
from torch import nn
from functools import reduce

import sklearn
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelBinarizer

import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

import tensorflow
import sys

<h1 id="dataset" style="color:#a97828; background:#4dc5ea;"> 
    <center>Dataset
        <a class="anchor-link" href="#dataset" target="_self">¶</a>
    </center>
</h1>

In [ ]:
path = '../input/creditcardfraud/creditcard.csv'
df = pd.read_csv(path, sep=",", index_col=None)
df.head()

In [ ]:
# Standardize
df['Amount'] = StandardScaler().fit_transform(df['Amount'].values.reshape(-1, 1))
df['Time'] = StandardScaler().fit_transform(df['Time'].values.reshape(-1, 1))

In [ ]:
anomalies = df[df["Class"] == 1]
normal = df[df["Class"] == 0]

anomalies.shape, normal.shape

In [ ]:
for f in range(0, 20):
    normal = normal.iloc[np.random.permutation(len(normal))]
    

data_set = pd.concat([normal[:2000], anomalies])

x_train, x_test = train_test_split(data_set, test_size = 0.4, random_state = 42)

x_train = x_train.sort_values(by=['Time'])
x_test = x_test.sort_values(by=['Time'])

y_train = x_train["Class"]
y_test = x_test["Class"]

x_train.head(10)

In [ ]:
x_train = np.array(x_train).reshape(x_train.shape[0], x_train.shape[1])
x_test = np.array(x_test).reshape(x_test.shape[0], x_test.shape[1])
input_shape = (x_train.shape[1], 1)

y_train = keras.utils.to_categorical(y_train, 2)
y_test = keras.utils.to_categorical(y_test, 2)

print("Shapes:\nx_train:%s\ny_train:%s\n" % (x_train.shape, y_train.shape))
print("x_test:%s\ny_test:%s\n" % (x_test.shape, y_test.shape))
print("input_shape:{}\n".format(input_shape))

<h1 id="tree" style="color:#a97828; background:#4dc5ea;"> 
    <center>Decision Tree
        <a class="anchor-link" href="#tree" target="_self">¶</a>
    </center>
</h1>

In [ ]:
class Tree():
    def __init__(self):
        self.num_cut = [1, 1]
        self.num_leaf = np.prod(np.array(self.num_cut) + 1)
        self.num_class = 2
        
    def torch_kron_prod(self, a, b):
        res = torch.einsum('ij,ik->ijk', [a, b])
        res = torch.reshape(res, [-1, np.prod(res.shape[1:])])
        return res
    
    def torch_bin(self, x, cut_points, temperature=0.1):
        D = cut_points.shape[0]
        W = torch.reshape(torch.linspace(1.0, D + 1.0, D + 1), [1, -1])
        cut_points, _ = torch.sort(cut_points)
        b = torch.cumsum(torch.cat([torch.zeros([1]), -cut_points], 0),0)
        h = torch.matmul(x, W) + b
        res = torch.exp(h-torch.max(h))
        res = res/torch.sum(res, dim=-1, keepdim=True)
        return h
    
    def nn_decision_tree(self, x, cut_points_list, leaf_score, temperature=0.1):
        leaf = reduce(self.torch_kron_prod,
                      map(lambda z: self.torch_bin(x[:, z[0]:z[0] + 1], z[1], temperature), enumerate(cut_points_list)))
        return torch.matmul(leaf, leaf_score)

In [ ]:
tree = Tree()

cut_points_list = [torch.rand([i], requires_grad=True) for i in tree.num_cut]
leaf_score = torch.rand([tree.num_leaf, tree.num_class], requires_grad=True)

loss_function = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cut_points_list + [leaf_score], lr=0.001, weight_decay=0.001)

In [ ]:
model = nn.Sequential(
          nn.Linear(31,16),
          nn.ReLU(),
          nn.Linear(16,8),
          nn.ReLU(),
          nn.Linear(8,2),
          nn.Sigmoid()
        )

<h1 id="training" style="color:#a97828; background:#4dc5ea;"> 
    <center>Training
        <a class="anchor-link" href="#training" target="_self">¶</a>
    </center>
</h1>

In [ ]:
for i in range(300):
    optimizer.zero_grad()
    x_batches = torch.split(torch.tensor(x_train).type(torch.float32), 32)
    y_batches = torch.split(torch.tensor(y_train).type(torch.long), 32)
    
    losses = torch.zeros(len(x_batches))
    accs = torch.zeros(len(x_batches))
    for j,x in enumerate(x_batches):
        out = model(x)
        y_pred = tree.nn_decision_tree(out, cut_points_list, 
                                       leaf_score, temperature=0.1)
        y_max = torch.max(y_batches[j], axis=1)[1]
        y_pred_max = torch.max(y_pred, axis=1)[1]
        
        acc = len(torch.where(y_max == y_pred_max)[0]) / len(y_max)
        accs[j] = acc
        
        loss = loss_function(y_pred, y_max)
        losses[j] = loss
        
        loss.backward()
        optimizer.step()
    if((i+1) % 20 == 0):
        print("i:{:4d}, loss:{:1.3f}, acc:{:1.3f}".format(i+1, losses.mean(), accs.mean()))

<h1 id="prediction" style="color:#a97828; background:#4dc5ea;"> 
    <center>Prediction
        <a class="anchor-link" href="#prediction" target="_self">¶</a>
    </center>
</h1>

In [ ]:
def predict(tree, X, y):
    X = torch.tensor(X).type(torch.float32)
    y = torch.tensor(y).type(torch.long)

    out = model(X)
    y_pred = tree.nn_decision_tree(out, cut_points_list, leaf_score, temperature=0.1)
    y_max = torch.max(y, axis=1)[1]
    y_pred_max = torch.max(y_pred, axis=1)[1]

    acc = len(torch.where(y_max == y_pred_max)[0]) / len(y_max)
    loss = loss_function(y_pred, y_max)
    return y_pred, acc, loss

In [ ]:
y_pred, acc, loss = predict(tree, x_test, y_test)
print("Accuracy:{:1.3f}, Loss:{:1.3f}"
         .format(acc, loss))

<h1 id="visualization" style="color:#a97828; background:#4dc5ea;"> 
    <center>Visaulization
        <a class="anchor-link" href="#visualization" target="_self">¶</a>
    </center>
</h1>

In [ ]:
class Visualization:
    labels = ["Normal", "Anomaly"]

    def draw_confusion_matrix(self, y, ypred):
        matrix = confusion_matrix(y, ypred)

        plt.figure(figsize=(10, 8))
        colors=[ "orange","green"]
        sns.heatmap(matrix, xticklabels=self.labels, yticklabels=self.labels, cmap=colors, annot=True, fmt="d")
        plt.title("Confusion Matrix")
        plt.ylabel('Actual')
        plt.xlabel('Predicted')
        plt.show()


    def draw_anomaly(self, y, error, threshold):
        groupsDF = pd.DataFrame({'error': error,
                                 'true': y}).groupby('true')

        figure, axes = plt.subplots(figsize=(12, 8))

        for name, group in groupsDF:
            axes.plot(group.index, group.error, marker='x' if name == 1 else 'o', linestyle='',
                    color='r' if name == 1 else 'g', label="Anomaly" if name == 1 else "Normal")

        axes.hlines(threshold, axes.get_xlim()[0], axes.get_xlim()[1], colors="b", zorder=100, label='Threshold')
        axes.legend()
        
        plt.title("Anomalies")
        plt.ylabel("Error")
        plt.xlabel("Data")
        plt.show()

    def draw_error(self, error, threshold):
            plt.plot(error, marker='o', ms=3.5, linestyle='',
                     label='Point')

            plt.hlines(threshold, xmin=0, xmax=len(error)-1, colors="b", zorder=100, label='Threshold')
            plt.legend()
            plt.title("Reconstruction error")
            plt.ylabel("Error")
            plt.xlabel("Data")
            plt.show()

In [ ]:
visualize = Visualization()
y_pred2 = torch.max(y_pred, axis=1)[1].detach().numpy()
y_test2 = np.argmax(y_test, axis=1)
visualize.draw_confusion_matrix(y_test2, y_pred2)